In [ ]:
import numpy as np
import pandas as pd

# ========= 1) Excel 读列 =========
file_path = r"your_data.xlsx"
sheet_name = "Eval"
indicator_cols = ["c1","c2","c3","c4"]
df = pd.read_excel(file_path, sheet_name=sheet_name)
X = df[indicator_cols].to_numpy(dtype=float)

# ========= 2) 参数模板 =========
params = {}  # 熵权法核心参数较少

P = X / np.maximum(np.sum(X, axis=0, keepdims=True), 1e-12)
n = X.shape[0]
k = 1.0 / np.log(n)
e = -k * np.sum(P * np.log(np.maximum(P, 1e-12)), axis=0)
d = 1 - e
w = d / np.maximum(np.sum(d), 1e-12)
print(w)


# 熵权法

## 输入说明

- 数据文件：默认读取脚本同目录下的 `data.csv`，也可以在代码顶部把 `DATA_FILE` 改为 `.xlsx` 或绝对路径。
- 数据格式：一般要求“一行一个样本/时刻/方案，一列一个变量/指标”。具体列名需要在代码顶部的 `TODO` 参数区填写。
- 示例：若模型需要特征 `特征1、特征2` 和目标列 `y`，表格可整理为：

| 特征1 | 特征2 | y |
|---:|---:|---:|
| 1.2 | 3.4 | 8.1 |
| 2.0 | 2.8 | 9.0 |

## 输出说明

- 控制台会打印核心结果，例如模型参数、评价指标、最优解、排名或预测值。
- 默认结果保存到代码顶部 `OUTPUT_FILE` 指定的文件。
- 若模型包含图形分析，会额外输出图片文件，例如箱型图 `boxplot.png`。

## 原理通俗解释

熵权法 的核心思想是：先把实际问题抽象成可计算的数据结构，再用对应的数学规则寻找“预测值、分类结果、综合得分或最优方案”。代码中已经保留主要计算流程，比赛时重点是把题目数据整理成表格，并把 TODO 参数替换为题目含义一致的列名和约束。

## 适用场景

客观确定多指标权重。

## 局限性

指标波动越大权重越高，不一定等于业务重要性更高。

## 使用提示

- 运行前先检查缺失值、异常值和量纲；很多模型对数据尺度敏感。
- 所有 `TODO` 都应结合题目背景填写，不要直接使用示例列名。
- 建模论文中建议同时写明参数来源，例如权重来自 AHP/熵权法，预测步数来自题目要求。

In [ ]:
"""
熵权法

使用方法：
1. 按照下方 TODO 修改 DATA_FILE、列名、参数和输出文件名。
2. 将数据文件放在本脚本同目录，或把 DATA_FILE 改成绝对路径。
3. 运行：python "熵权法.py"
"""

from pathlib import Path
import numpy as np
import pandas as pd



DATA_FILE = "data.csv"  # TODO: 请填写[数据文件路径]，说明：CSV/Excel 均可；若使用 Excel，请在 load_data 中改为 read_excel。
OUTPUT_FILE = "model_output.csv"  # TODO: 请填写[输出文件名]，说明：保存模型结果，建议保留 .csv 或 .xlsx 后缀。
RANDOM_STATE = 42  # TODO: 请填写[随机种子]，说明：用于复现实验；整数即可。
INDICATOR_COLUMNS = ["指标1", "指标2"]  # TODO: 请填写[指标列名列表]，说明：数值型。
NEGATIVE_COLUMNS = []  # TODO: 请填写[负向指标列名]，说明：越小越好的指标填入此列表。



REQUIRES_DATA = True  # 参数型模型可不提供数据文件；表格型模型必须提供数据。


def load_data() -> pd.DataFrame:
    """读取用户数据；竞赛时通常把 Excel/CSV 表格整理成一行一个样本。"""
    path = Path(DATA_FILE)
    if not path.exists():
        if not REQUIRES_DATA:
            return pd.DataFrame()
        raise FileNotFoundError(
            f"未找到数据文件 {DATA_FILE}。请先修改 DATA_FILE，或将数据放到脚本同目录。"
        )
    if path.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    return pd.read_csv(path)


def run_model(data: pd.DataFrame) -> None:
    X = data[INDICATOR_COLUMNS].astype(float).copy()
    for col in NEGATIVE_COLUMNS:
        X[col] = X[col].max() - X[col]
    X = (X - X.min()) / (X.max() - X.min()) + 1e-12
    P = X / X.sum(axis=0)
    e = -(P * np.log(P)).sum(axis=0) / np.log(len(X))
    d = 1 - e
    weights = d / d.sum()
    result = pd.DataFrame({"指标": INDICATOR_COLUMNS, "熵权": weights.values})
    result.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
    print(result)


if __name__ == "__main__":
    df = load_data()
    run_model(df)
